## EDA With full poem list

- Maybe actually just sentiment analysis? Text clustering? Something like that?
    - Remove all the common words from each article and then plot all the words with connections between them or something 

In [26]:
import pandas as pd
import re
from plotnine import *
import polars as pl

#### Read in my data and see the dtypes

In [27]:
df = pd.read_csv('data/poets_full_dataset.csv')
df.dtypes

Title         object
Poet          object
Year         float64
URL           object
Poem Text     object
dtype: object

There are about 120 poems with years missing. This comes from the site itself (the few poems I checked do not have years listed online either), so I will just filter them out (because they're less than 1% of my dataset)

In [28]:
df = df[df['Year'].notna()]

Now I can make them type int instead of float

In [29]:
df['Year'] = df['Year'].astype('int64')

That looks better

In [30]:
df.dtypes

Title        object
Poet         object
Year          int64
URL          object
Poem Text    object
dtype: object

The scraper/poems used "\n" to denote line breaks sometimes. I don't think that I want these to be included in the text moving forwards. I thought about adding a `line_breaks` column, but it seems like not every line break is represented as a "\n", so it wouldn't be terribly accurate I think.

<hr>

### Remove audio-only poems

In [31]:
df = df[~df['Title'].str.contains('audio only')]

<hr>

Instead, I'm just replacing "\n" with a " ".

In [32]:
df['Poem Text'] = df['Poem Text'].str.replace("\n", " ")

In [33]:
df.head()

,Title,Poet,Year,URL,Poem Text
0,A Line-storm Song,Robert Frost,1913,https://poets.org/poem/line-storm-song,"The line-storm clouds fly tattered and swift, ..."
1,The Weary Blues,Langston Hughes,1926,https://poets.org/poem/weary-blues,"Droning a drowsy syncopated tune, Rocking back..."
2,Morning in the Burned House,Margaret Atwood,1995,https://poets.org/poem/morning-burned-house,In the burned house I am eating breakfast. You...
3,On Living,Nâzim Hikmet,1994,https://poets.org/poem/living,Living is no laughing matter: you must live wi...
4,I Could Be a Whale Shark,Aimee Nezhukumatathil,2018,https://poets.org/poem/i-could-be-whale-shark,"Bolinao, Philippines I am worried about tentac..."


<hr>

#### For further analysis, I think it would be helpful for all the words to be formatted the same, and I do not need punctuation here either

I define a function **with the assistance of Claude** (to account for some edge cases when I'm removing punctuation, like an apostrophe within a word that should not be replaced with a spcae, but an em-dash within two words that should be replaced by a space)

I then manually checked some entries to make sure all the rules are applying as I would expect

In [34]:
def clean_text(text):
    if not isinstance(text, str):
        return text
    # Rule 1: apostrophe (ASCII or curly) between word characters → delete
    text = re.sub(r"(?<=\w)['\u2019](?=\w)", "", text)
    # Rule 2: other special character between two word characters → replace with space
    text = re.sub(r"(?<=\w)[^\w\s](?=\w)", " ", text)
    # Rule 3: remaining special characters (next to a space or boundary) → delete
    text = re.sub(r"[^\w\s]", "", text)
    # Collapse any double spaces left behind
    text = re.sub(r" {2,}", " ", text).strip()
    return text

In [35]:
df['pt_nopunct'] = df['Poem Text'].str.lower().apply(clean_text)

Making sure there are no double spaces in my rows

In [36]:
df[df['Poem Text'].str.contains("  ", na=False)]

,Title,Poet,Year,URL,Poem Text,pt_nopunct


In [37]:
df[df['pt_nopunct'].str.contains("  ", na=False)]

,Title,Poet,Year,URL,Poem Text,pt_nopunct


Also adding a column that is the length of each poem in words, just in case that sort of thing would be helpful later

In [38]:
df['str_len'] = df['pt_nopunct'].str.split().str.len()

In [39]:
df.to_csv('data/small_w_pl.csv', index=False)

<hr>

### Making a hugeeee dataframe where each poem is a row and each column is one word of the poem

Like 16587 rows and 15879 columns

In [40]:
df_long = df['pt_nopunct'].str.split(' ', expand=True)
df = df.join(df_long)

In [41]:
df_long.shape

(16110, 15879)

Was using this cell to export a few poems for testing. I would:
- Export the individual row as a csv
- Copy the text of the poem from the web version to a google doc and manually apply my punctuation removal rules
- Copy the text of the exported poem from the csv into the google sheet on the next page
- Command+F with the full text I manually corrected and make sure it matched 1-to-1 with the code-cleaned text

In [42]:
df_test_1 = df[df['Title']=="The Weary Blues"]
df_test_1.to_csv('data/test_1.csv', index=False)

In [43]:
df.head()

,Title,Poet,Year,URL,Poem Text,pt_nopunct,str_len,0,1,2,...,15869,15870,15871,15872,15873,15874,15875,15876,15877,15878
0,A Line-storm Song,Robert Frost,1913,https://poets.org/poem/line-storm-song,"The line-storm clouds fly tattered and swift, ...",the line storm clouds fly tattered and swift t...,220.0,the,line,storm,...,None,None,None,None,None,None,None,None,None,None
1,The Weary Blues,Langston Hughes,1926,https://poets.org/poem/weary-blues,"Droning a drowsy syncopated tune, Rocking back...",droning a drowsy syncopated tune rocking back ...,283.0,droning,a,drowsy,...,None,None,None,None,None,None,None,None,None,None
2,Morning in the Burned House,Margaret Atwood,1995,https://poets.org/poem/morning-burned-house,In the burned house I am eating breakfast. You...,in the burned house i am eating breakfast you ...,231.0,in,the,burned,...,None,None,None,None,None,None,None,None,None,None
3,On Living,Nâzim Hikmet,1994,https://poets.org/poem/living,Living is no laughing matter: you must live wi...,living is no laughing matter you must live wit...,144.0,living,is,no,...,None,None,None,None,None,None,None,None,None,None
4,I Could Be a Whale Shark,Aimee Nezhukumatathil,2018,https://poets.org/poem/i-could-be-whale-shark,"Bolinao, Philippines I am worried about tentac...",bolinao philippines i am worried about tentacl...,238.0,bolinao,philippines,i,...,None,None,None,None,None,None,None,None,None,None


In [44]:
df.to_csv('data/df_wide.csv', index=False)

<hr>

### Pivoting long (eek!)

I originally tried a simple df.melt(), which you can see below commented out. It did not work and it killed my kernel. I think it might've been because it was 263,000,000 cells.

In [45]:
# value_cols = [i for i in range(0, 15879)]

In [46]:
# df_long = df.melt(
#     id_vars=['Title', 'Poet', 'Year', 'URL', 'Poem Text', 'pt_nopunct', 'str_len'], 
#     value_vars=value_cols, 
#     var_name='word_num', 
#     value_name='word'
# )

I had **Claude Code** help me with this, using polars instead of pandas and saving the file as a parquet.

I don't know 100% all that it's doing, but I understand the `pl.scan_csv()` is different from `pl.read_csv()`, and that otherwise the `.melt()` is pretty similar to Pandas. The `schema_overrides` came because it was reading one of the columns as boolean and not as a string, so **Claude wrote** that line to manually define the schemas for all my single-word columns as string.

In [47]:
id_cols = ['Title', 'Poet', 'Year', 'URL', 'Poem Text', 'pt_nopunct', 'str_len']

schema_overrides = {str(i): pl.String for i in range(15879)}



(
    pl.scan_csv('data/df_wide.csv', schema_overrides=schema_overrides)
    .unpivot(index=id_cols, variable_name='position', value_name='word')
    .with_columns(pl.col('position').cast(pl.Int32))
    .sink_parquet('data/df_long.parquet')
)

Then, since I love Pandas, I brought the parquet back in as a df, but I filtered out words that were "None" as I did so. The "None" filler was created when I expanded each poem's text so it was 1 word per column, and it is a string and not an NaN because, I think, they were coerced to strings when I saved as csv and then scanned and saved as parquet

I also am no longer bringing in the "Poem Text" column because I tried that, and then I realized that the final .csv was over 12GB. Because if was storing the full poem text for each row of the dataset, so like roughly 135 extra words for each of the 3 million rows...

In [48]:
df_long_fp = pd.read_parquet(
    'data/df_long.parquet',
    columns=['Title', 'Poet', 'Year', 'URL', 'position', 'word'],
    filters=[('word', '!=', "None")]
)

In [49]:
df_long_fp = df_long_fp.sort_values(by=['Year', 'Title', 'position'], ascending=[True, True, True])

In [50]:
df_long_fp.to_csv('data/df_long.csv', index=False)